In [4]:
import fitz  # PyMuPDF
import re
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

# ----------------------------
# 1. Extract raw text
# ----------------------------
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text")
    return text

# ----------------------------
# 2. Parse metadata
# ----------------------------
def parse_metadata(text):
    metadata = {}

    # Title - take full case name after "Case Details"
    title_match = re.search(r"Case Details\s*\n([\s\S]*?)\n\n", text)
    if title_match:
        metadata["title"] = title_match.group(1).strip()

    # Judges - clean up *, whitespace
    judges_match = re.search(r"\[(.*?)JJ?\.\]", text)
    if judges_match:
        judges = [j.strip().replace("*","") for j in judges_match.group(1).split("and")]
        metadata["judges"] = judges

    # Keywords
    keywords_match = re.search(r"List of Keywords\n(.*?)\n\n", text, re.DOTALL)
    if keywords_match:
        metadata["keywords"] = [k.strip() for k in keywords_match.group(1).split(";")]

    # Sections / Acts - remove duplicates
    acts_match = list(set(re.findall(r"Article \d+|IPC \d+|CrPC \d+", text)))
    metadata["sections"] = acts_match

    # Year
    year_match = re.search(r"\[(\d{4})\]", text)
    if year_match:
        metadata["year"] = int(year_match.group(1))

    # Court
    metadata["court"] = "Supreme Court of India"

    return metadata


# ----------------------------
# 3. Chunk judgment body
# ----------------------------
def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " "]
    )
    return splitter.split_text(text)

# ----------------------------
# 4. Setup ChromaDB
# ----------------------------
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name="precedent_finder")
except:
    pass  # ignore if it doesn't exist

collection = chroma_client.create_collection(
    name="precedent_finder",
    embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)


# ----------------------------
# 5. Ingest one file
# ----------------------------
def ingest_case(pdf_path, case_id):
    raw_text = extract_text_from_pdf(pdf_path)
    metadata = parse_metadata(raw_text)
    # before adding to Chroma
    metadata["judges"] = ", ".join(metadata.get("judges", []))
    metadata["keywords"] = ", ".join(metadata.get("keywords", []))
    metadata["sections"] = ", ".join(metadata.get("sections", []))

    # Print metadata before inserting
    print("\n📑 Extracted Metadata:")
    for k, v in metadata.items():
        print(f"{k}: {v}")

    chunks = chunk_text(raw_text)

    for i, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            metadatas=[{
                **metadata,
                "case_id": case_id,
                "chunk_id": i
            }],
            ids=[f"{case_id}_{i}"]
        )

    print(f"\n✅ Ingested {len(chunks)} chunks from {case_id}")

# ----------------------------
# 6. Run on single PDF
# ----------------------------
pdf_path = r"cases\2024-1-case-1.pdf"   # change path as needed
case_id = "2024-1-case-1"
ingest_case(pdf_path, case_id)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



📑 Extracted Metadata:
judges: J.K. Maheshwari, K.V. Viswanathan,
sections: Article 142
year: 2024
court: Supreme Court of India
keywords: 


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



✅ Ingested 31 chunks from 2024-1-case-1


In [5]:
query = "trivial errors in recruitment applications"
results = collection.query(query_texts=[query], n_results=3)

print("\n🔎 Search Results:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("\n---")
    print(doc[:200], "...")
    print("Metadata:", meta)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔎 Search Results:

---
13.	 Equally undisputed is the fact that after filling out the application, the 
appellant cleared the written examination and the Physical Eligibility 
Test. It was also stated in the counter affidav ...
Metadata: {'case_id': '2024-1-case-1', 'chunk_id': 14, 'court': 'Supreme Court of India', 'judges': 'J.K. Maheshwari, K.V. Viswanathan,', 'keywords': '', 'sections': 'Article 142', 'year': 2024}

---
following clause in the advertisement:-
“Instructions to fill online application form are available 
on the website. It is recommended to all the candidates 
to carefully read the instructions before  ...
Metadata: {'case_id': '2024-1-case-1', 'chunk_id': 15, 'court': 'Supreme Court of India', 'judges': 'J.K. Maheshwari, K.V. Viswanathan,', 'keywords': '', 'sections': 'Article 142', 'year': 2024}

---
wrong or mis-leading information – There is an exception for trivial 
errors or omissions as law does concern itself with trifles – This 
principle is recognized in t